In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler


1. Linear Regression Exercise:
   Using the California Housing dataset from scikit-learn, create a linear regression model to predict house prices.
   Evaluate the performance of Linear Regression on test set.

In [2]:
# Unable to load the dataset directly using housing = fetch_california_housing(). 
# Encountered HTTPError.
# Therefore, I downloaded the file to my local: https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv
# Load local dataset
df = pd.read_csv("../data/housing.csv")

This dataset comes from the California Census.
Each row = one census block group, which is:
- A small geographic area
- Contains many households and housing units

In [3]:
#Sanity Check: Compute average rooms per household
df['rooms_per_household'] = df['total_rooms'] / df['households']
average_rooms_per_household = df['rooms_per_household'].mean()
print(f"Average rooms per household: {average_rooms_per_household}")

Average rooms per household: 5.428999742190376


In [4]:
## Feature Engineering: 
 
# Due to high correlation between room-related variables, 
# rooms_per_household will be removed to reduce multicollinearity and improve coefficient interpretability.  

# bedrooms_per_household
df['bedrooms_per_household'] = df['total_bedrooms'] / df['households']
average_bedrooms_per_household = df['bedrooms_per_household'].mean()
print(f"Average bedrooms per household: {average_bedrooms_per_household}")

# population_per_household
df['population_per_household'] = df['population'] / df['households']
average_population_per_household = df['population_per_household'].mean()
print(f"Average population per household: {average_population_per_household}")

# ocean_proximity one-hot encoding
df["ocean_proximity"] = pd.Categorical(
    df["ocean_proximity"],
    categories=[
        "INLAND",
        "<1H OCEAN",
        "NEAR BAY",
        "NEAR OCEAN",
        "ISLAND"
    ],
    ordered=False
)
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)
# Show only ocean_proximity encoded columns
ocean_cols = [c for c in df.columns if c.startswith("ocean_proximity_")]
print("Encoded ocean_proximity columns:", ocean_cols)

# Drop rows with NaN/inf caused by divide-by-zero or missing values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=["bedrooms_per_household", "population_per_household"], inplace=True)


# Final feature set
feature_cols = [
    "housing_median_age",
    "bedrooms_per_household",
    "population_per_household",
    "median_income"
] + ocean_cols

X = df[feature_cols]
y = df["median_house_value"]

print("Final features used:", feature_cols)
print("X shape:", X.shape)

Average bedrooms per household: 1.0970623858069952
Average population per household: 3.0706551594363742
Encoded ocean_proximity columns: ['ocean_proximity_<1H OCEAN', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN', 'ocean_proximity_ISLAND']
Final features used: ['housing_median_age', 'bedrooms_per_household', 'population_per_household', 'median_income', 'ocean_proximity_<1H OCEAN', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN', 'ocean_proximity_ISLAND']
X shape: (20433, 8)


In [5]:
## Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

print("Train shape:", X_train.shape)
print("Test shape : ", X_test.shape)  

Train shape: (16346, 8)
Test shape :  (4087, 8)


In [ ]:
## Train-fit the model
model = LinearRegression()
# StandardScaler performs standardization:scaled value = (value - mean) / stddev
scaler = StandardScaler()
# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)
model.fit(X_train_scaled, y_train)

# Apply same transformation to test data
X_test_scaled = scaler.transform(X_test)
y_pred = model.predict(X_test_scaled)

## Evaluate the model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse}")
print(f"R^2: {r2}")

RMSE: 74433.7979756157
R^2: 0.594857986321685


In [7]:
## Inspect coefficients
coefficients = pd.Series(model.coef_, index=X.columns)
print("Model Coefficients:")
print(coefficients.sort_values(ascending=False))

Model Coefficients:
median_income                 72955.387377
ocean_proximity_<1H OCEAN     36432.835356
ocean_proximity_NEAR OCEAN    29891.748176
ocean_proximity_NEAR BAY      26350.017060
housing_median_age            12328.597350
bedrooms_per_household         5736.801408
ocean_proximity_ISLAND         4252.902567
population_per_household      -3676.275865
dtype: float64


Interpretation:
1. Median income is the strongest predictor in the model: Higher median household income is strongly associated with higher housing prices, reflecting greater purchasing power and demand.
2. Ocean Proximity (Location Effects): Convenient access to coastal amenities increases housing desirability. 
3. Housing Median Age: Older housing areas are associated with higher prices, potentially reflecting established neighborhoods and central locations.
4. Bedrooms per Household: Areas with more bedrooms per household tend to have higher housing values, indicating a preference for larger living spaces.
5. Population per Household: Higher population per household is negatively associated with housing prices, suggesting that crowding reduces housing desirability.

================================================================================
================================================================================
2. Classification Exercise:
   Using the breast cancer dataset from scikit-learn, build classification models to predict malignant vs benign tumors.
   Compare Logistic Regression and KNN performance on test set.

   ```python
   from sklearn.datasets import load_breast_cancer

   # Load dataset
   cancer = load_breast_cancer()

In [8]:
from sklearn.datasets import load_breast_cancer

# Load dataset
cancer = load_breast_cancer()
# Print the dataset
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
print("Target variable added to the dataset:")
print(cancer.target_names)
print("Breast Cancer Dataset:")
print(df.head())
print("Describing the Dataset:")
print(df.describe())

Target variable added to the dataset:
['malignant' 'benign']
Breast Cancer Dataset:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean 

In [9]:
X = cancer.data
y = cancer.target

In [ ]:
## Train-Test Split
# stratify=y keeps the malignant/benign proportion consistent in both datasets. 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# StandardScaler performs standardization:scaled value = (value - mean) / stddev
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## Train Logistic Regression model
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)
y_pred_log = log_reg.predict(X_test_scaled)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

## Train KNN model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# for k in range(1, 21):
#     knn = KNeighborsClassifier(n_neighbors=k)
#     scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
#     print(f"k={k}, mean accuracy={scores.mean():.3f}")

knn = KNeighborsClassifier(n_neighbors=8)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

Logistic Regression Accuracy: 0.9824561403508771
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

KNN Accuracy: 0.9736842105263158
              precision    recall  f1-score   support

           0       0.98      0.95      0.96        42
           1       0.97      0.99      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



Logistic Regression: 
- Achieves near-perfect performance with an accuracy of approximately 98%.
- Demonstrates balanced precision and recall across both malignant and benign classes.
- Results in only about 2% overall classification errors.
- Achieves a recall of 0.98 for malignant tumors, meaning that very few cancer cases are missed.

KNN
- Also demonstrates strong classification performance.
- Shows slightly lower overall accuracy compared to Logistic Regression.
- The key difference lies in its lower recall for malignant tumors, indicating a higher number of missed cancer cases.

For medical diagnostic purposes, Logistic Regression is more suitable due to its higher recall for malignant tumors, balanced performance across classes, and greater reliability in minimizing false negatives.